# W6D2 — An Autoencoder, and No Labels At All — Guided

**Week 6 · Day 2 · Representation Learning** · Lab

Every model you have trained since week 1 was shown the answer. This one is not. Today's network
sees 320 images, is told nothing about any of them, and is scored only on how well it puts back
what it was given.

You start from the four numbers on this morning's slide — `x = [1, 0, 1, 0]`, the encoder that
makes the code `[1.0, 1.0]`, and the reconstruction error **0.20**. Then you set the bottleneck to
four, use the identity, and get **0.0**: a perfect score, from a network that learned nothing.

The core is that same lesson at full size. You train at four bottleneck widths and watch
reconstruction error fall every time you widen it. Then you fit a classifier on the codes and watch
accuracy go the *other* way at the point where reconstruction looks best. Two curves, one axis, and
the gap between them is why nobody reports reconstruction error as a representation score.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٦ · اليوم ٢ — مُرمِّز ذاتي، وبلا تسميات إطلاقًا

**الأسبوع السادس · اليوم الثاني · تعلّم التمثيل** · معمل

كل نموذج درّبته منذ الأسبوع الأول رأى الجواب. وهذا لا يراه. فشبكة اليوم ترى ثلاثمئة وعشرين صورة،
ولا يُقال لها عن أيّ منها شيء، ولا تُقيَّم إلا على مدى إعادتها ما أُعطيت.

تبدأ من الأعداد الأربعة التي على شريحة هذا الصباح — `x = [1, 0, 1, 0]`، والمُرمِّز الذي يُنتج
الشيفرة `[1.0, 1.0]`، وخطأ إعادة البناء **٠٫٢٠**. ثم تجعل عنق الزجاجة أربعة وتستعمل المصفوفة
المحايدة فتحصل على **٠٫٠**: درجة كاملة من شبكة لم تتعلّم شيئًا.

والقسم الأساسي هو الدرس نفسه بالحجم الكامل. تُدرّب عند أربعة عروض لعنق الزجاجة وتراقب خطأ إعادة
البناء يهبط كلّما وسّعته. ثم تُلائم مصنّفًا على الشيفرات وتراقب الدقّة تذهب في الاتجاه **الآخر** عند
النقطة التي تبدو فيها إعادة البناء في أفضل حال. منحنيان على محور واحد، والفجوة بينهما هي سبب ألّا
أحد يذكر خطأ إعادة البناء درجةً للتمثيل.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Compute an autoencoder's code, reconstruction and error by hand, and say which of the three is
  the product.
- Build an encoder / decoder pair whose bottleneck width is a parameter, and assert that the
  decoder gives back the input's shape.
- Train a model with no labels anywhere in the loop, and prove it by inspecting the signature of
  your own training function.
- Sweep the bottleneck width and read two curves against each other — reconstruction error and
  downstream accuracy — instead of one.
- Say why a wide bottleneck gives you the best reconstruction score and a worse representation.
- Use a code space for nearest-neighbour search, and read the failures as information.
- Say what a denoising objective adds, in one sentence, and what reconstruction error is and is not
  usable for out of distribution.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تحسب شيفرة مُرمِّزٍ ذاتي وإعادة بنائه وخطأه باليد، وأن تقول أيّ الثلاثة هو المنتَج.
- أن تبني زوج مُرمِّز ومُفكِّك ترميز يكون عرض عنق الزجاجة فيه معاملًا، وأن تفحص أن المُفكِّك يُعيد شكل
  الدخل.
- أن تُدرّب نموذجًا بلا تسمية في أيّ موضع من الحلقة، وأن تُبرهن ذلك بفحص توقيع دالة تدريبك نفسها.
- أن تمسح عرض عنق الزجاجة وتقرأ منحنيين أحدهما مقابل الآخر — خطأ إعادة البناء والدقّة اللاحقة —
  بدل منحنى واحد.
- أن تقول لماذا يمنحك عنق الزجاجة الواسع أفضل درجة إعادة بناء وتمثيلًا أسوأ.
- أن تستعمل فضاء الشيفرات في البحث عن الجيران الأقرب، وأن تقرأ الإخفاقات معلوماتٍ لا أخطاء.
- أن تقول في جملة ما الذي يضيفه هدف إزالة الضوضاء، وفيمَ يصلح خطأ إعادة البناء وفيمَ لا يصلح خارج
  التوزيع.

</div>

## About the data

`small_image_5class` again — the same 400 images as yesterday and as W4D4 and W4D5, five classes,
COCO CC-BY. Today they are **downscaled to 64×64**, which is not a detail: at 224×224 a
convolutional autoencoder does not finish inside the slot on a laptop CPU, and at 64×64 the whole
sweep takes about a hundred seconds. One image is therefore `64 × 64 × 3` = **12,288 numbers**, and
that number is the widest bottleneck in the sweep for a reason.

**The labels are loaded, and they are not used for training.** They are used twice: to fit the
evaluation probe in task 2.4, and to colour the nearest-neighbour results in task 2.5. Both are
*measurements taken afterwards*. No label reaches the optimiser, and task 2.2 asserts it by
inspecting the training function's own signature rather than by promising.

**The known problem** is the one from week 4: twelve of the eighty `pizza` images are
near-duplicates. Today it matters more than it did yesterday — the train/val split here is a plain
stratified split, so a duplicate pair straddling it makes the probe look slightly better than it
is. That is a real limitation of today's numbers, and Thursday fixes it by reusing W4D5's
duplicate-grouped split.

**Compute:** four trainings at 100 epochs each — roughly **14 s, 14 s, 15 s and 50 s** on a laptop
CPU, so about a hundred seconds in total, plus twenty more for the denoising run. No downloads.

<div dir="rtl" align="right">

## عن البيانات

`small_image_5class` مرّةً أخرى — الصور الأربعمئة نفسها التي رأيتها أمس وفي الأسبوع الرابع، خمس
فئات، من مجموعة COCO المرخّصة. وهي اليوم **مُصغَّرة إلى ٦٤×٦٤**، وليس هذا تفصيلًا: فعند ٢٢٤×٢٢٤ لا
ينتهي المُرمِّز الالتفافي داخل الحصّة على معالج حاسوب محمول، وعند ٦٤×٦٤ يستغرق المسح كله نحو مئة
ثانية. فالصورة الواحدة إذًا `64 × 64 × 3` = **١٢٬٢٨٨** عددًا، وهذا العدد هو أوسع عنق زجاجة في المسح
لسبب.

**التسميات مُحمَّلة، وهي غير مستعملة في التدريب.** تُستعمل مرّتين: لملاءمة فحص التقييم في المهمة ٢٫٤،
ولتلوين نتائج الجيران الأقرب في المهمة ٢٫٥. وكلتاهما **قياس بعد الحدث**. فلا تصل تسمية إلى المُحسِّن،
وتفحص المهمة ٢٫٢ ذلك بتفتيش توقيع دالة التدريب نفسها لا بالوعد.

**والمشكلة المعروفة** هي مشكلة الأسبوع الرابع: اثنتا عشرة من صور `pizza` الثمانين شبه مكرّرة. وهي
اليوم أهمّ ممّا كانت أمس — فالتقسيم هنا تقسيم طبقيّ عادي، وزوجٌ مكرّر يتوزّع بين شطريه يجعل الفحص
يبدو أفضل ممّا هو. وهذا قيد حقيقي على أرقام اليوم، ويُصلحه الخميس بإعادة استخدام تقسيم الأسبوع
الرابع المُجمَّع بالتكرارات.

**الحساب:** أربع عمليات تدريب بمئة دورة لكلٍّ — نحو **١٤ و١٤ و١٥ و٥٠ ثانية** على معالج حاسوب محمول،
أي نحو مئة ثانية إجمالًا، وعشرون أخرى لتشغيلة إزالة الضوضاء. ولا تنزيلات.

</div>

## Setup

No new libraries. The images are loaded once into a single tensor — 400 × 3 × 64 × 64 float32 is
about 20 MB, so there is no reason for a `DataLoader` today and one less moving part in the
training loop.

<div dir="rtl" align="right">

## الإعداد

لا مكتبات جديدة. وتُحمَّل الصور مرّةً واحدة في موتّر واحد — ‏٤٠٠ × ٣ × ٦٤ × ٦٤ بدقّة `float32` نحو
عشرين ميجابايت، فلا داعي لـ`DataLoader` اليوم، وذلك جزءٌ متحرّك أقلّ في حلقة التدريب.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset_dir
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report
from aiep.viz import use_course_style, savefig

ensure("torch", "scikit-learn", "matplotlib", "pandas", "pyarrow")
seed_everything(42)

import inspect
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

use_course_style()
np.set_printoptions(precision=4, suppress=True)

SEED = 42
IMAGE_SIZE = 64
INPUT_DIM = IMAGE_SIZE * IMAGE_SIZE * 3          # 12,288 — the widest bottleneck in the sweep
TRUNK_DIM = 8 * 8 * 16                           # what the conv stack hands the bottleneck
EPOCHS = 100
BATCH_SIZE = 32
BOTTLENECKS = (4, 32, 256, INPUT_DIM)

IMAGE_ROOT = get_dataset_dir("small_image_5class") / "images"
PATHS = sorted(IMAGE_ROOT.rglob("*.jpg"))
LABELS = np.array([p.parent.name for p in PATHS])
CLASSES = sorted(set(LABELS))


def load_images(paths, size=IMAGE_SIZE):
    """Every image as one float tensor in [0, 1], shape (n, 3, size, size)."""
    array = np.stack([np.asarray(Image.open(p).convert("RGB").resize((size, size)),
                                 dtype=np.float32) / 255.0 for p in paths])
    return torch.from_numpy(array).permute(0, 3, 1, 2).contiguous()


IMAGES = load_images(PATHS)
TRAIN_INDEX, VAL_INDEX = train_test_split(np.arange(len(PATHS)), test_size=0.2,
                                          stratify=LABELS, random_state=SEED)

RECON_DIR = ARTEFACT_DIR / "recon"
RECON_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(PATHS)} images {tuple(IMAGES.shape)} | classes {CLASSES}")
print(f"train {len(TRAIN_INDEX)} | val {len(VAL_INDEX)} | one image = {INPUT_DIM:,} numbers")
print(versions(), "| device:", device())

## Section 1 — Warm-up: four numbers, two bottlenecks  (≈25 min)

Everything here works. This is the worked example from the slides, with the lecture's matrices.

The encoder is 2 × 4, the decoder is 4 × 2, and the input is `x = [1, 0, 1, 0]`. Row 1 of the
encoder against `x` is `0.5 − 0 + 0.5 − 0` = 1.0; row 2 is `0.5 + 0 + 0.5 + 0` = 1.0. The code is
**`[1.0, 1.0]`** — four numbers became two. Decoding gives **`[1.0, 0.1, 1.0, 0.1]`**, and the total
absolute error against `x` is **0.20**.

Then the second cell does the thing that matters. Set the bottleneck to **four** — as wide as the
input — take the identity for both matrices, and the error is **exactly 0.0**. A perfect score on
the objective, from a model that learned the identity function.

Change `x` and re-run both cells. The narrow one's error moves. The wide one's stays at zero, for
every input you can invent, which is the whole problem with it.

*(A note on the number: the morning's spec sheet quotes 0.3 for this example. With the one-decimal
matrices actually shown on the slide it is 0.20, the slides use the computed value, and so does
this lab.)*

<div dir="rtl" align="right">

## القسم الأول — الإحماء: أربعة أعداد وعنقا زجاجة (نحو ٢٥ دقيقة)

كل ما هنا يعمل. وهذا المثال المحلول من الشرائح بمصفوفات المحاضرة.

المُرمِّز ٢×٤، والمُفكِّك ٤×٢، والدخل `x = [1, 0, 1, 0]`. والصف الأول من المُرمِّز مقابل `x` هو
`0.5 − 0 + 0.5 − 0` = ١٫٠، والصف الثاني `0.5 + 0 + 0.5 + 0` = ١٫٠. فالشيفرة **`[1.0, 1.0]`** — أربعة
أعداد صارت اثنين. ويُعطي فكّ الترميز **`[1.0, 0.1, 1.0, 0.1]`**، ومجموع الخطأ المطلق مقابل `x`
هو **٠٫٢٠**.

ثم تفعل الخلية الثانية ما يهمّ. اجعل عنق الزجاجة **أربعة** — بعرض الدخل — وخذ المصفوفة المحايدة
للاثنين، فيكون الخطأ **صفرًا تمامًا**. درجة كاملة على الهدف، من نموذج تعلّم دالة الهوية.

غيّر `x` وأعِد تشغيل الخليتين. يتحرّك خطأ الضيّق. ويبقى خطأ الواسع صفرًا لأيّ دخل تخترعه، وهذه هي
المشكلة كلها.

*(ملاحظة على الرقم: تذكر ورقة مواصفات الصباح ‎٠٫٣‎ لهذا المثال. وهو بالمصفوفات ذات المنزلة الواحدة
المعروضة على الشريحة ‎٠٫٢٠‎، والشرائح تستعمل القيمة المحسوبة، وكذلك هذا المعمل.)*

</div>

In [ ]:
# The slide's encoder, decoder and input.
E = np.array([[0.5, -0.5, 0.5, -0.5],
              [0.5, 0.5, 0.5, 0.5]])
D = np.array([[0.5, 0.5],
              [-0.4, 0.5],
              [0.5, 0.5],
              [-0.4, 0.5]])
x = np.array([1.0, 0.0, 1.0, 0.0])

code = E @ x
reconstruction = D @ code
NARROW_ERROR = float(np.abs(reconstruction - x).sum())

print(f"x              : {x}")
print(f"code (2 wide)  : {code}")
print(f"reconstruction : {reconstruction}")
print(f"absolute error : |{reconstruction[0]:.1f}-1| + |{reconstruction[1]:.1f}-0| + "
      f"|{reconstruction[2]:.1f}-1| + |{reconstruction[3]:.1f}-0| = {NARROW_ERROR:.2f}")

# And the point of the whole day: similar inputs get nearby codes, with no labels anywhere.
x2 = np.array([1.0, 0.0, 1.0, 1.0])      # one value different
x3 = np.array([0.0, 1.0, 0.0, 1.0])      # every value different
print(f"\ncode of x2 (differs in 1 place) : {E @ x2}   distance {np.linalg.norm(code - E @ x2):.3f}")
print(f"code of x3 (differs in 4 places): {E @ x3}   distance {np.linalg.norm(code - E @ x3):.3f}")

In [ ]:
# Now widen the bottleneck to the input's own width and take the identity for both halves.
E_wide = np.eye(4)
D_wide = np.eye(4)

wide_code = E_wide @ x
wide_reconstruction = D_wide @ wide_code
WIDE_ERROR = float(np.abs(wide_reconstruction - x).sum())

print(f"x                   : {x}")
print(f"code (4 wide)       : {wide_code}   <- a copy of the input")
print(f"reconstruction      : {wide_reconstruction}")
print(f"absolute error      : {WIDE_ERROR:.2f}")
print(f"\nnarrow bottleneck: {NARROW_ERROR:.2f}   wide bottleneck: {WIDE_ERROR:.2f}")
print("By the only number we were measuring, the second model is perfect.")
print("It learned the identity function. There was no pressure to find structure,")
print("because there was no shortage of room.")

## Section 2 — Core: six tasks  (≈60 min)

1. A conv encoder / decoder with the bottleneck width as a parameter.
2. Train at `bottleneck=32` with no labels, and look at the reconstructions.
3. The sweep: 4, 32, 256, 12,288.
4. Fit a probe on the codes, and put accuracy on the same axis as reconstruction error.
5. Nearest neighbours in code space.
6. Denoising: corrupt the input, keep the clean target.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. مُرمِّز ومُفكِّك التفافيان بعرض عنق الزجاجة معاملًا.
٢. درّب عند `bottleneck=32` بلا تسميات، وانظر إلى إعادة البناء.
٣. المسح: ٤ و٣٢ و٢٥٦ و١٢٬٢٨٨.
٤. لائم فحصًا على الشيفرات، وضع الدقّة على محور خطأ إعادة البناء نفسه.
٥. الجيران الأقرب في فضاء الشيفرات.
٦. إزالة الضوضاء: أفسِد الدخل واحتفظ بالهدف النظيف.

</div>

### Task 2.1 — the architecture, with the bottleneck as a parameter

Three strided convolutions take `3 × 64 × 64` down to `16 × 8 × 8` — that is 1,024 numbers — and a
linear layer takes those 1,024 to whatever `bottleneck` says. The decoder is the mirror image:
linear back to 1,024, reshape to `16 × 8 × 8`, three transposed convolutions back up to
`3 × 64 × 64`, and a sigmoid because the pixels live in `[0, 1]`.

The one thing to assert is that the decoder's output shape equals the input's. It is the cheapest
possible test and it catches every off-by-one in the transposed convolutions, which is the single
most common bug in this architecture — an output of `63 × 63` or `66 × 66` broadcasts against
nothing and raises an error much later, in the loss, with a message about sizes that names neither
layer.

<div dir="rtl" align="right">

### المهمة ٢٫١ — البنية، وعرض عنق الزجاجة معاملًا

ثلاث طبقات التفاف بخطوة تُنزل `3 × 64 × 64` إلى `16 × 8 × 8` — أي ١٬٠٢٤ عددًا — وطبقة خطّية تأخذ
هذه الألف وأربعة وعشرين إلى ما يقوله `bottleneck`. والمُفكِّك صورةٌ مرآتية: خطّية إلى ١٬٠٢٤، وإعادة
تشكيل إلى `16 × 8 × 8`، وثلاث التفافات منقولة صعودًا إلى `3 × 64 × 64`، ودالّة `sigmoid` لأن
البكسلات في المجال `[0, 1]`.

والشيء الوحيد الذي يجب فحصه أن شكل خرج المُفكِّك يساوي شكل الدخل. وهو أرخص فحص ممكن ويُمسك كل انزلاق
بواحد في الالتفافات المنقولة، وهو أشيع خلل في هذه البنية — فخرجٌ بمقاس `63 × 63` أو `66 × 66` لا
يُبثّ على شيء ويرفع خطأً متأخّرًا في دالّة الخسارة برسالةٍ عن المقاسات لا تُسمّي أيًّا من الطبقتين.

</div>

In [ ]:
class Autoencoder(nn.Module):
    """Conv encoder, linear bottleneck of `bottleneck` numbers, mirrored decoder."""

    def __init__(self, bottleneck):
        super().__init__()
        # TODO: Build self.encoder and self.decoder as described above.
        # مهمة: ابنِ `self.encoder` و`self.decoder` كما وُصف أعلاه.

    def forward(self, images):
        codes = self.encoder(images)
        return self.decoder(codes), codes


# TODO: Build one autoencoder at bottleneck 32, run two images through it, and print both shapes together with whether they match.
# مهمة: ابنِ مُرمِّزًا واحدًا عند عنق زجاجة ٣٢، ومرّر صورتين، واطبع الشكلين مع بيان تطابقهما.

### Task 2.2 — train it, and prove no label was involved

The training loop is the one you already know with one thing removed: there is no `y`.

Write `train_autoencoder(images, bottleneck, epochs)`. It takes images and returns a trained model
and its loss history. **It does not take labels, and that is checkable** — the sanity check at the
bottom reads the function's signature with `inspect` and fails if a parameter called `y`, `labels`
or `targets` appears in it. That is a deliberately literal-minded test, and it is there because
"self-supervised" is a claim about the loop, not a mood.

Train at `bottleneck=32` for 100 epochs — about 14 seconds — plot the loss, then display eight
original / reconstruction pairs.

**The reconstructions are blurry.** That is expected, not a bug: mean squared error over pixels is
minimised by predicting the average of everything plausible, and the average of several plausible
sharp textures is a smooth one. Write that sentence down, because on Wednesday the MAE's
reconstructions are blurry for exactly the same reason and it is worth having said it once already.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — درّبه، وبرهِن أن لا تسمية دخلت

حلقة التدريب هي التي تعرفها بشيء واحد محذوف: لا يوجد `y`.

اكتب `train_autoencoder(images, bottleneck, epochs)`. تأخذ صورًا وتُعيد نموذجًا مُدرَّبًا وتاريخ
خسارته. **ولا تأخذ تسميات، وهذا قابل للفحص** — ففحص السلامة في الأسفل يقرأ توقيع الدالة بـ`inspect`
ويفشل إن ظهر فيه معامل اسمه `y` أو `labels` أو `targets`. وهو فحص حرفيّ عن قصد، وهو هناك لأن
«التعلّم ذاتي الإشراف» ادّعاء عن الحلقة لا مزاج.

درّب عند `bottleneck=32` مئة دورة — نحو أربع عشرة ثانية — وارسم الخسارة، ثم اعرض ثمانية أزواج من
الأصل وإعادة البناء.

**إعادة البناء ضبابية.** وهذا متوقّع لا خلل: فالخطأ التربيعي المتوسّط على البكسلات يصغر عند التنبّؤ
بمتوسّط كل ما هو محتمل، ومتوسّط عدّة نُسج حادّة محتملة نسيجٌ ناعم. دوّن هذه الجملة، فإعادة بناء
المُرمِّز المُقنَّع يوم الأربعاء ضبابية للسبب نفسه بالضبط، ويحسن أن تكون قد قلتَها مرّة.

</div>

In [ ]:
def train_autoencoder(images, bottleneck, epochs=EPOCHS, noise=0.0):
    """Train an autoencoder on `images`. No labels, by construction — check the signature."""
    # TODO: Seed, build the model, and run the Adam loop with the batch as its own target. When `noise` is non-zero, add Gaussian noise to the input only, never to the target.
    # مهمة: ثبّت البذرة، وابنِ النموذج، وشغّل حلقة `Adam` بالدفعة هدفًا لنفسها. وحين لا يكون `noise` صفرًا أضف ضوضاء غاوسية إلى الدخل وحده لا إلى الهدف أبدًا.


@torch.no_grad()
def encode_all(model, images):
    """Codes and per-image reconstruction MSE for every image."""
    model.eval()
    reconstructed, codes = model(images)
    error = ((reconstructed - images) ** 2).mean(dim=(1, 2, 3))
    return codes.numpy(), error.numpy(), reconstructed


# TODO: Train at bottleneck 32 on the training images only, plot the loss curve, and show eight original / reconstruction pairs from the validation set.
# مهمة: درّب عند عنق زجاجة ٣٢ على صور التدريب وحدها، وارسم منحنى الخسارة، واعرض ثمانية أزواج من الأصل وإعادة البناء من مجموعة التحقّق.

### Task 2.3 — the bottleneck sweep

Train the same architecture at **4, 32, 256 and 12,288**. The last one is the full input dimension:
`64 × 64 × 3`. Nothing has to be discarded at that width, and the network will not discard anything
it does not have to.

You already have 32 from the previous task — do not retrain it, reuse it. Record for each width the
mean reconstruction MSE on the validation images, the training time and the parameter count.

Record it twice — on the training images and on the validation images — because the two columns do
not say the same thing.

The expected shape: **training reconstruction error falls every time you widen the bottleneck**, and
it falls by a lot between 4 and 32. That is the curve everybody plots, and read on its own it says
"wider is better". The validation column falls too, and then **turns back up at 12,288**: with 25
million parameters and 320 images the widest model is partly memorising, and memorisation does not
transfer to images it did not see.

The full-width run takes about 50 seconds — it has 25 million parameters, almost all of them in the
two bottleneck matrices.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — مسح عنق الزجاجة

درّب البنية نفسها عند **٤ و٣٢ و٢٥٦ و١٢٬٢٨٨**. والأخير هو بُعد الدخل الكامل: `64 × 64 × 3`. فلا شيء
يجب طرحه عند هذا العرض، ولن تطرح الشبكة ما لا يلزمها طرحه.

وعندك ٣٢ من المهمة السابقة — فلا تُعِد تدريبه، أعِد استعماله. وسجّل لكل عرض متوسّط خطأ إعادة البناء
على صور التحقّق، وزمن التدريب، وعدد المعاملات.

وسجّله مرّتين — على صور التدريب وعلى صور التحقّق — فالعمودان لا يقولان الشيء نفسه.

والشكل المتوقّع: **يهبط خطأ إعادة البناء على التدريب كلّما وسّعت العنق**، ويهبط كثيرًا بين ٤ و٣٢. وهذا
المنحنى الذي يرسمه الجميع، وهو مقروءًا وحده يقول «الأوسع أفضل». ويهبط عمود التحقّق أيضًا ثم **يرتدّ
صاعدًا عند ١٢٬٢٨٨**: فبخمسة وعشرين مليون معامل وثلاثمئة وعشرين صورة يحفظ النموذج الأوسع جزءًا من
بياناته، والحفظ لا ينتقل إلى صور لم يرها.

وتستغرق تشغيلة العرض الكامل نحو خمسين ثانية — ففيها خمسة وعشرون مليون معامل، جُلّها في مصفوفتَي
عنق الزجاجة.

</div>

In [ ]:
# TODO: Train at every width in BOTTLENECKS, reusing the bottleneck-32 model, and collect width, train and validation reconstruction MSE, seconds and parameter count into a DataFrame.
# مهمة: درّب عند كل عرض في `BOTTLENECKS` مع إعادة استعمال نموذج الـ٣٢، واجمع العرض وخطأَي إعادة البناء على التدريب والتحقّق والثواني وعدد المعاملات في `DataFrame`.

### Task 2.4 — the second curve, and the divergence

Now put a number on the thing you actually want.

For each width, take the codes, fit a **linear** classifier on the training images' codes, and
score it on the validation images'. **The labels are used here and only here, for evaluation.**
Nothing about the autoencoders changes; they were trained before this cell ran and they are not
touched by it. If you are ever unsure whether a pipeline is self-supervised, this is the boundary
to look for: labels after the encoder is frozen are a measurement, labels inside the loop are
supervision.

Plot both curves against bottleneck width, on twin axes.

They do not track each other. The 256-wide bottleneck reconstructs **better** than the 32-wide one
and its codes classify **worse** — and that pair of facts, on one picture, is the entire lab. The
full-width run reconstructs best of all, and it is not the best representation either.

Why: reconstruction rewards keeping whatever predicts pixels, and most of what predicts pixels is
local texture and colour that says nothing about which of five objects is in the frame. Narrow the
bottleneck and the network cannot afford texture; it has to spend its few numbers on something more
summary. **The constraint is not a cost of the method. The constraint is the method.**

<div dir="rtl" align="right">

### المهمة ٢٫٤ — المنحنى الثاني، والتباعد

ضع الآن رقمًا على الشيء الذي تريده فعلًا.

لكل عرض، خذ الشيفرات، ولائم مصنّفًا **خطيًا** على شيفرات صور التدريب، وقيّمه على شيفرات صور التحقّق.
**والتسميات تُستعمل هنا وهنا فقط، للتقييم.** فلا شيء يتغيّر في المُرمِّزات؛ فقد دُرِّبت قبل تشغيل هذه
الخلية ولا تمسّها. وإن شككتَ يومًا في كون مسارٍ ذاتيَّ الإشراف فهذا هو الحدّ الذي تبحث عنه: التسميات
بعد تجميد المُرمِّز قياس، والتسميات داخل الحلقة إشراف.

ارسم المنحنيين مقابل عرض عنق الزجاجة على محورين متقابلين.

وهما لا يتتبّعان أحدهما الآخر. فعنق الزجاجة ذو الـ٢٥٦ يُعيد البناء **أفضل** من ذي الـ٣٢ وشيفراته
تُصنّف **أسوأ** — وهذان الأمران معًا في صورة واحدة هما المعمل كله. وتشغيلة العرض الكامل تُعيد البناء
أفضل من الجميع، وليست أفضل تمثيل أيضًا.

والسبب: إعادة البناء تكافئ الاحتفاظ بما يتنبّأ بالبكسلات، ومعظم ما يتنبّأ بالبكسلات نسيجٌ ولونٌ
محلّيان لا يقولان شيئًا عن أيّ الأجسام الخمسة في الإطار. وضيّق العنق فلا تقدر الشبكة على النسيج؛
فتُنفق أعدادها القليلة على شيء أكثر إجمالًا. **والقيد ليس كلفةً للطريقة. القيد هو الطريقة.**

</div>

In [ ]:
def linear_probe(codes, train_index=TRAIN_INDEX, val_index=VAL_INDEX):
    """Validation accuracy of a linear classifier on these codes. Evaluation only."""
    pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    pipeline.fit(codes[train_index], LABELS[train_index])
    return float(pipeline.score(codes[val_index], LABELS[val_index]))


# TODO: Score every width's codes with linear_probe, add the column to SWEEP, plot both curves on twin axes, and state the 32-against-256 comparison in words.
# مهمة: قيّم شيفرات كل عرض بـ`linear_probe`، وأضف العمود إلى `SWEEP`، وارسم المنحنيين على محورين متقابلين، واذكر مقارنة ٣٢ مقابل ٢٥٦ بالكلمات.

### Task 2.5 — nearest neighbours in code space

A representation is a map on which distance means something. Test that directly.

Take five validation images as queries, and for each one find the four nearest of the *other*
images by cosine similarity on the 32-dim codes. Display the query and its four neighbours, and
label each with its class.

Then measure it properly. Five queries is a picture, not a result, so also compute the hit rate
over **every** validation image: roughly a third of the retrieved neighbours share their query's
class, against a fifth by chance. Better than nothing and a long way from good — which is the
honest description of a 32-dim autoencoder code, and the number Thursday's DINOv2 features have to
beat.

The misses are the interesting part. A dog on grass retrieving a zebra on grass is not a random
error; it tells you the code is carrying background, which is precisely the "texture and colour"
failure mode from the previous task, now visible as a picture instead of a number.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — الجيران الأقرب في فضاء الشيفرات

التمثيل خريطةٌ للمسافة عليها معنى. افحص ذلك مباشرةً.

خذ خمس صور من التحقّق مُستعلِمات، وجِد لكلٍّ منها أقرب أربع من الصور **الأخرى** بجيب تمام الزاوية على
الشيفرات ذات الـ٣٢ بُعدًا. واعرض المُستعلِم وجيرانه الأربعة، وسمِّ كلًّا بفئته.

ثم قِس الأمر قياسًا صحيحًا. فخمسة مُستعلِمات صورةٌ لا نتيجة، فاحسب أيضًا نسبة الإصابة على **كل** صور
التحقّق: نحو ثلث الجيران المُسترجَعين يشاركون مُستعلِمهم فئته، مقابل الخُمس بالمصادفة. أفضل من لا شيء
وبعيدٌ عن الجيّد — وهذا هو الوصف الصادق لشيفرة مُرمِّزٍ ذاتي بـ٣٢ بُعدًا، وهو الرقم الذي على تمثيلات
DINOv2 يوم الخميس أن تتجاوزه.

والإخفاقات هي الجزء المثير. فاسترجاع كلبٍ على العشب لحمار وحشيّ على العشب ليس خطأً عشوائيًا؛ إنه
يخبرك أن الشيفرة تحمل الخلفية، وهو بعينه إخفاق «النسيج واللون» من المهمة السابقة، ظاهرًا الآن صورةً
بدل رقم.

</div>

In [ ]:
# TODO: Find the 4 nearest neighbours by cosine on the standardised 32-dim codes for 5 query images, display them in a grid, then report the hit rate over all validation queries.
# مهمة: جِد أقرب أربعة جيران بجيب التمام على الشيفرات الموحّدة المقاييس لخمس صور مُستعلِمة، واعرضها في شبكة، ثم أبلغ عن نسبة الإصابة على كل مُستعلِمات التحقّق.

### Task 2.6 — denoising

One change to the pretext task: add Gaussian noise to the **input** and keep the **clean** image as
the target. Everything else stays.

Your `train_autoencoder` already takes a `noise` argument for this. Retrain at bottleneck 32 with
`noise=0.2`, then feed it noisy validation images and display three rows: noisy input, the
denoiser's output, and the clean original. The output is visibly cleaner than what went in — the
network was never asked to reproduce the noise and had no way to, so it learned to reproduce
everything except.

Then the sentence to write: **why does making the task harder make the representation better?**
Because copying stops working. A plain autoencoder can lower its loss by passing information
straight through; a denoising one cannot, because the thing it would be passing through is partly
noise. It is forced to model what the image is *likely* to be, and that is a statement about
structure rather than about pixels. This is the same argument Wednesday makes with masking, at a
much more aggressive setting.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — إزالة الضوضاء

تغيير واحد في المهمة الذريعة: أضف ضوضاء غاوسية إلى **الدخل** واحتفظ بالصورة **النظيفة** هدفًا. وكل
ما عدا ذلك يبقى.

ودالّتك `train_autoencoder` تأخذ وسيط `noise` لهذا أصلًا. أعِد التدريب عند عنق زجاجة ٣٢ بـ
`noise=0.2`، ثم أطعِمه صور تحقّق مشوّشة واعرض ثلاثة صفوف: الدخل المشوّش، وخرج مُزيل الضوضاء، والأصل
النظيف. والخرج أنظف بوضوح ممّا دخل — فالشبكة لم تُطلب منها الضوضاء ولا سبيل لها إليها، فتعلّمت أن
تُعيد كل شيء عداها.

ثم الجملة التي تكتبها: **لماذا يجعل تصعيبُ المهمة التمثيلَ أفضل؟** لأن النسخ يكفّ عن العمل. فالمُرمِّز
العادي يستطيع خفض خسارته بتمرير المعلومة كما هي؛ ومُزيل الضوضاء لا يستطيع، لأن ما سيُمرّره ضوضاء في
جزء منه. فيُضطرّ إلى نمذجة ما **يُرجَّح** أن تكون عليه الصورة، وهذا قولٌ عن البنية لا عن البكسلات. وهي
الحجّة نفسها التي يسوقها الأربعاء بالإخفاء، عند إعدادٍ أعنف بكثير.

</div>

In [ ]:
NOISE = 0.2

# TODO: Retrain at bottleneck 32 with noise=NOISE, denoise eight validation images, show the three rows, and print the input-vs-clean and output-vs-clean MSEs.
# مهمة: أعِد التدريب عند عنق زجاجة ٣٢ بـ`noise=NOISE`، وأزِل الضوضاء عن ثماني صور تحقّق، واعرض الصفوف الثلاثة، واطبع خطأ الدخل والخرج مقابل النظيف.

## Section 3 — Stretch: reconstruction error as an anomaly signal  (≈30 min)

Autoencoders are genuinely shipped for anomaly detection, on the theory that an input unlike
anything in training reconstructs badly. Test the theory.

1. Rank all 400 images by the bottleneck-32 model's per-image reconstruction error and display the
   eight worst and the eight best. Look at them before you form an opinion.
2. Then load three images from `unseen_images` whose class the autoencoder has never seen — the
   elephants, marked `in_or_out = out` in that dataset's `labels.csv` — and compute their error.

Is the elephants' error higher than the in-distribution images'? Higher than the worst in-class
ones? Write two sentences on whether reconstruction error is a usable out-of-distribution signal
**on this data**, and say what you would need — more data, a threshold, a different score — before
you would put it in front of a user.

This is the honest version of the technique. A method that works on MNIST and does not work on 400
COCO crops has not been debunked; it has been scoped, and knowing the scope is the skill.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: خطأ إعادة البناء إشارةً على الشذوذ (نحو ٣٠ دقيقة)

تُستعمل المُرمِّزات الذاتية فعلًا في كشف الشذوذ، على أساس أن دخلًا لا يشبه شيئًا في التدريب يُعاد بناؤه
بسوء. افحص هذه النظرية.

١. رتّب الصور الأربعمئة بخطأ إعادة البناء لكل صورة من نموذج الـ٣٢، واعرض أسوأ ثماني وأفضل ثماني.
   وانظر إليها قبل أن تُكوّن رأيًا.
٢. ثم حمّل ثلاث صور من `unseen_images` من فئة لم يرها المُرمِّز قط — الأفيال، الموسومة
   `in_or_out = out` في ملف `labels.csv` لتلك المجموعة — واحسب خطأها.

أخطأ الأفيال أعلى من خطأ صور التوزيع؟ أعلى من أسوأ الصور داخل الفئات؟ اكتب جملتين عن صلاحية خطأ
إعادة البناء إشارةً على الخروج عن التوزيع **على هذه البيانات**، وقل ما الذي تحتاجه — بيانات أكثر، أو
عتبة، أو درجة مختلفة — قبل أن تضعه أمام مستخدم.

وهذه هي النسخة الصادقة من التقنية. فطريقة تعمل على MNIST ولا تعمل على أربعمئة قصاصة من COCO لم
تُدحض؛ بل حُدِّد نطاقها، ومعرفة النطاق هي المهارة.

</div>

In [ ]:
# TODO: Show the 8 worst and 8 best reconstructed images, then score the three out-of-class unseen_images and compare their error against the in-distribution distribution.
# مهمة: اعرض أسوأ ثماني صور وأفضل ثماني في إعادة البناء، ثم قيّم صور `unseen_images` الثلاث الخارجة عن الفئات وقارن خطأها بتوزيع الخطأ الداخلي.

**Your two sentences.** Replace this line: is reconstruction error a usable out-of-distribution
signal on this data, and what would you need before shipping it?

*(What to look at when you answer: the count on the last line. A threshold is only useful if
in-distribution images almost never cross it, and the number of in-class images sitting above the
elephants' mean error tells you directly how many false alarms that threshold would raise.)*

<div dir="rtl" align="right">

**جملتاك.** استبدل هذا السطر: هل يصلح خطأ إعادة البناء إشارةً على الخروج عن التوزيع على هذه
البيانات، وما الذي تحتاجه قبل إطلاقه؟

*(ما تنظر إليه وأنت تجيب: العدد في السطر الأخير. فالعتبة لا تنفع إلا إن كانت صور التوزيع لا تكاد
تتجاوزها، وعدد الصور داخل الفئات الجالسة فوق متوسّط خطأ الأفيال يخبرك مباشرةً بكم إنذارًا كاذبًا
سترفعه تلك العتبة.)*

</div>

## Save your artefact

`ae_codes.parquet` — one row per image: the file, its class, its 32-dim code and its reconstruction
error. **Thursday loads this file** and puts the same linear probe on these codes as on DINOv2's
features, on identical folds, in one table. Today's codes are one of the four rows in that table,
and they are the row you built yourself.

Also saved: `recon/` with the loss curve, the reconstruction pairs, the sweep, the neighbours, the
denoising grid and the anomaly ranking.

<div dir="rtl" align="right">

## احفظ أثرك

`ae_codes.parquet` — صفٌّ لكل صورة: الملف، وفئته، وشيفرته ذات الـ٣٢ بُعدًا، وخطأ إعادة بنائه.
**ويُحمّل الخميس هذا الملف** فيضع الفحص الخطيّ نفسه على هذه الشيفرات وعلى تمثيلات DINOv2، على
التقسيمات نفسها، في جدول واحد. وشيفرات اليوم صفٌّ من صفوف ذلك الجدول الأربعة، وهي الصف الذي بنيته
بنفسك.

ويُحفظ أيضًا `recon/` وفيه منحنى الخسارة، وأزواج إعادة البناء، والمسح، والجيران، وشبكة إزالة
الضوضاء، وترتيب الشذوذ.

</div>

In [ ]:
codes_frame = pd.DataFrame({
    "file": [f"{p.parent.name}/{p.name}" for p in PATHS],
    "label": LABELS,
    "split": np.where(np.isin(np.arange(len(PATHS)), TRAIN_INDEX), "train", "val"),
    "recon_mse": ERROR_32,
})
for dimension in range(CODES_32.shape[1]):
    codes_frame[f"code_{dimension:02d}"] = CODES_32[:, dimension]

CODES_PATH = ARTEFACT_DIR / "ae_codes.parquet"
codes_frame.to_parquet(CODES_PATH, index=False)

SWEEP.to_parquet(ARTEFACT_DIR / "ae_sweep.parquet", index=False)

print(codes_frame.iloc[:5, :6].to_string(index=False))
print(f"\n{len(codes_frame)} rows x {codes_frame.shape[1]} columns -> {CODES_PATH.name}")
print(f"{len(list(RECON_DIR.glob('*.png')))} figures in {RECON_DIR.name}/")
print("Thursday loads ae_codes.parquet as one row of the comparison table.")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check_close(NARROW_ERROR, 0.20,
            f"the warm-up's narrow autoencoder must reproduce the slide's 0.20 — got "
            f"{NARROW_ERROR:.4f}. Check the decoder matrix: the -0.4 entries are what make the "
            f"error 0.20 rather than 0",
            f"يجب أن يُعيد المُرمِّز الضيّق في الإحماء رقم الشريحة ‎٠٫٢٠‎ — والناتج {NARROW_ERROR:.4f}. "
            f"وافحص مصفوفة المُفكِّك: مدخلتا ‎−٠٫٤‎ هما ما يجعل الخطأ ‎٠٫٢٠‎ لا صفرًا",
            tol=1e-9)

check_close(WIDE_ERROR, 0.0,
            f"the identity bottleneck must reconstruct exactly — got {WIDE_ERROR:.4f}. This is the "
            f"perfect score that comes from learning nothing",
            f"يجب أن يُعيد عنق الزجاجة المحايد البناء تمامًا — والناتج {WIDE_ERROR:.4f}. وهذه هي "
            f"الدرجة الكاملة الآتية من ألّا تتعلّم شيئًا",
            tol=1e-12)

check(SHAPE_PRESERVED,
      f"the decoder's output shape must equal the input's — got {tuple(sample_out.shape)} from "
      f"{tuple(sample_batch.shape)}. An off-by-one here surfaces much later, inside the loss",
      f"يجب أن يساوي شكل خرج المُفكِّك شكل الدخل — والناتج {tuple(sample_out.shape)} من "
      f"{tuple(sample_batch.shape)}. وانزلاقٌ بواحد هنا يظهر متأخّرًا داخل دالّة الخسارة")

check(not ({"y", "labels", "targets", "target"} & set(inspect.signature(train_autoencoder).parameters)),
      f"train_autoencoder must take no labels — its parameters are "
      f"{list(inspect.signature(train_autoencoder).parameters)}. Self-supervised is a property of "
      f"the loop, and this is the check that makes it one",
      f"يجب ألّا تأخذ `train_autoencoder` تسميات — ومعاملاتها "
      f"{list(inspect.signature(train_autoencoder).parameters)}. فالإشراف الذاتي خاصّية للحلقة، "
      f"وهذا هو الفحص الذي يجعله كذلك")

check(RECON_FALLS,
      f"training reconstruction error must fall at every widening of the bottleneck — got "
      f"{SWEEP.recon_train.round(5).tolist()} at widths {SWEEP.bottleneck.tolist()}",
      f"يجب أن يهبط خطأ إعادة البناء على التدريب عند كل توسيع لعنق الزجاجة — والناتج "
      f"{SWEEP.recon_train.round(5).tolist()} عند العروض {SWEEP.bottleneck.tolist()}")

check(BETTER_RECON_WORSE_PROBE and BEST_RECON_WIDTH != BEST_PROBE_WIDTH,
      f"this is the lab's thesis: the 256-wide bottleneck must reconstruct better than the 32-wide "
      f"one and probe worse, and the best-reconstructing width must not be the best-representing "
      f"one — recon {SWEEP.recon_train.round(5).tolist()}, accuracy "
      f"{SWEEP.probe_accuracy.round(3).tolist()} at widths {SWEEP.bottleneck.tolist()}",
      f"هذه أطروحة المعمل: يجب أن يُعيد عنق الزجاجة ذو الـ٢٥٦ البناء أفضل من ذي الـ٣٢ وأن يكون فحصه "
      f"أسوأ، وألّا يكون أفضل عرضٍ في إعادة البناء أفضلَ عرضٍ في التمثيل — إعادة البناء "
      f"{SWEEP.recon_train.round(5).tolist()}، والدقّة {SWEEP.probe_accuracy.round(3).tolist()} عند "
      f"العروض {SWEEP.bottleneck.tolist()}")

check(NEIGHBOUR_HIT_RATE > CHANCE,
      f"nearest neighbours in code space must beat chance over the validation set — got "
      f"{NEIGHBOUR_HIT_RATE:.3f} against {CHANCE:.2f}. At chance the code carries no class "
      f"information at all and nothing downstream can recover it",
      f"يجب أن يتجاوز الجيران الأقرب في فضاء الشيفرات المصادفةَ على مجموعة التحقّق — والناتج "
      f"{NEIGHBOUR_HIT_RATE:.3f} مقابل {CHANCE:.2f}. فعند المصادفة لا تحمل الشيفرة معلومة فئة "
      f"البتّة ولا يستطيع أيّ لاحقٍ استخراجها")

check(DENOISED_MSE < NOISY_MSE,
      f"the denoiser's output must be closer to the clean image than its noisy input was — got "
      f"{DENOISED_MSE:.5f} against {NOISY_MSE:.5f}",
      f"يجب أن يكون خرج مُزيل الضوضاء أقرب إلى الصورة النظيفة من دخله المشوّش — والناتج "
      f"{DENOISED_MSE:.5f} مقابل {NOISY_MSE:.5f}")

check(len(codes_frame) == len(PATHS) and codes_frame.filter(like="code_").shape[1] == 32,
      f"ae_codes.parquet must hold one row per image and 32 code columns — got "
      f"{len(codes_frame)} rows and {codes_frame.filter(like='code_').shape[1]} code columns",
      f"يجب أن يحمل `ae_codes.parquet` صفًّا لكل صورة و٣٢ عمود شيفرة — والناتج "
      f"{len(codes_frame)} صفًّا و{codes_frame.filter(like='code_').shape[1]} عمود شيفرة")

report()

## What's next

**W6D3 — masked autoencoders.** Tomorrow keeps today's idea and changes the corruption. Instead of
adding noise to every pixel, it **deletes 75% of the patches** and asks a model to put them back —
and 75% is not a typo. You will find the ratio at which the image genuinely stops being
recoverable, and it is far higher than anyone guesses.

You need two things from today and yesterday: `patch_check.json` from W6D1, which tomorrow's
warm-up reloads and masks by hand, and the fact you just wrote down about why a harder pretext task
gives a better encoder. Tomorrow is that sentence, taken to its extreme.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٦ اليوم ٣ — المُرمِّزات الذاتية المُقنَّعة.** الغد يُبقي فكرة اليوم ويُغيّر الإفساد. فبدل
إضافة ضوضاء إلى كل بكسل، **يحذف ٧٥٪ من الرقع** ويطلب من نموذج إعادتها — وليست الـ٧٥٪ خطأً مطبعيًا.
وستجد النسبة التي تكفّ عندها الصورة عن كونها قابلة للاسترجاع فعلًا، وهي أعلى بكثير ممّا يخمّن أحد.

وتحتاج من اليوم ومن أمس شيئين: ملف `patch_check.json` من الاثنين، ويُعيد إحماء الغد تحميله ويُخفيه
بيده، والحقيقة التي دوّنتها للتوّ عن سبب إعطاء المهمة الذريعة الأصعب مُرمِّزًا أفضل. فالغد هو تلك
الجملة مأخوذةً إلى أقصاها.

</div>